# MediGraph — Medical Knowledge Graph with Neo4j + LLM

> **Final Project — Graph Database (Semester 6)**
>
> Notebook ini membangun **Knowledge Graph** penyakit–gejala–obat–diet–olahraga–pencegahan menggunakan **Neo4j**, lalu mengintegrasikan **LLM (Large Language Model)** untuk analisis dan tanya-jawab cerdas.

---

## Arsitektur & Fitur

| Tier | Komponen | Deskripsi |
|:----:|----------|-----------|
| 1 | **Text-to-Cypher** | LLM menerjemahkan pertanyaan bahasa alami -> query Cypher |
| 1 | **Graph Analytics** | PageRank, Community Detection (Louvain), Shortest Path, Degree Centrality |
| 2 | **ML on Graph** | FastRP Node Embeddings + KNN Similarity Search |
| 3 | **LLM Graph Builder** | Ekstraksi entitas & relasi dari teks mentah -> populate Neo4j |
| 4 | **Graph-Augmented RAG** | Retrieve konteks dari graph -> augment prompt -> LLM jawab |

### Dataset (6 CSV)
- `Diseases_and_Symptoms_dataset.csv` — Binary symptom indicators per penyakit
- `description.csv` — Deskripsi setiap penyakit
- `medications.csv` — Daftar obat per penyakit
- `precautions.csv` — Tindakan pencegahan
- `workout.csv` — Rekomendasi olahraga
- `diets.csv` — Rekomendasi diet

### Entitas (Nodes) & Relasi (Edges)
```
(Disease) -[:HAS_SYMPTOM]-> (Symptom)
(Disease) -[:TREATED_WITH]-> (Medication)
(Disease) -[:HAS_PRECAUTION]-> (Precaution)
(Disease) -[:RECOMMENDED_WORKOUT]-> (Workout)
(Disease) -[:RECOMMENDED_DIET]-> (Diet)
```

---
## Cell 1 — Install Dependencies
Install semua library yang dibutuhkan.

In [ ]:
!pip install neo4j openai pandas -q
print("Semua dependensi berhasil di-install!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.1 MB/s eta 0:00:00
Semua dependensi berhasil di-install!


---
## Cell 2 — Konfigurasi API Key & Koneksi

**Setup OpenRouter:**
1. Buka **Settings** (ikon ️) di sidebar kiri Colab
2. Pilih **Secrets** -> tambahkan key:
   - `OPENROUTER_API_KEY` = API key dari [openrouter.ai](https://openrouter.ai)
3. Isi kredensial **Neo4j** di cell berikut

In [ ]:
import os, time, json, re
from google.colab import userdata

# OpenRouter API Key
try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except Exception:
    pass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = "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"

from openai import OpenAI, RateLimitError

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

def tanya(pertanyaan, model="meta-llama/llama-3.3-70b-instruct", retries=5, delay=10):
    """Query LLM with retry mechanism."""
    for i in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": pertanyaan}]
            )
            return response.choices[0].message.content
        except RateLimitError:
            print(f"Rate limited... retry {i+1}/{retries}, tunggu {delay} detik")
            time.sleep(delay)
    return "Gagal setelah semua retry."

def tanya_system(system_prompt, user_prompt, model="meta-llama/llama-3.3-70b-instruct", retries=5, delay=10):
    """Query LLM with system prompt."""
    for i in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ]
            )
            return response.choices[0].message.content
        except RateLimitError:
            print(f"Rate limited... retry {i+1}/{retries}, tunggu {delay} detik")
            time.sleep(delay)
    return "Gagal setelah semua retry."

# Neo4j credentials
NEO4J_URI      = "neo4j://127.0.0.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "emriq123"

# Jika mau pakai Secrets juga:
# NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")

# Test LLM connection
print("Testing koneksi OpenRouter...")
test_result = tanya("Jawab dalam satu kalimat: apa itu knowledge graph?")
print(f"LLM Response: {test_result}")

Testing koneksi OpenRouter...
LLM Response: Knowledge graph adalah sebuah struktur data yang menggambarkan hubungan antara rescued entitas yang kemudian digunakan untuk pertanyaan yang lebih kompleks dan penemuan pengetahuan baru atau berbasis hubungan.


---
## Cell 3 — Upload Dataset (6 CSV)

Klik tombol **"Choose Files"** yang muncul di bawah, lalu pilih ke-6 file CSV:
1. `Diseases_and_Symptoms_dataset.csv`
2. `description.csv`
3. `medications.csv`
4. `precautions.csv`
5. `workout.csv`
6. `diets.csv`

In [ ]:
from google.colab import files
import io

print("Silakan upload 6 file CSV dataset...")
print("   Files: Diseases_and_Symptoms_dataset.csv, description.csv, medications.csv,")
print("          precautions.csv, workout.csv, diets.csv\n")

uploaded = files.upload()

print(f"\n{len(uploaded)} file berhasil di-upload:")
for fname, content in uploaded.items():
    size_kb = len(content) / 1024
    print(f"   {fname} ({size_kb:.1f} KB)")

Silakan upload 6 file CSV dataset...
   Files: Diseases_and_Symptoms_dataset.csv, description.csv, medications.csv,
          precautions.csv, workout.csv, diets.csv



Saving description.csv to description.csv
Saving diets.csv to diets.csv
Saving Diseases_and_Symptoms_dataset.csv to Diseases_and_Symptoms_dataset.csv
Saving medications.csv to medications.csv
Saving precautions.csv to precautions.csv
Saving workout.csv to workout.csv

6 file berhasil di-upload:
   description.csv (18.5 KB)
   diets.csv (20.1 KB)
   Diseases_and_Symptoms_dataset.csv (44890.8 KB)
   medications.csv (16.3 KB)
   precautions.csv (10.8 KB)
   workout.csv (20.6 KB)


---
##  Cell 4 — Load & Preview Data
Membaca semua CSV ke dalam pandas DataFrame dan menampilkan preview.

In [ ]:
import pandas as pd
import ast

# Load dataset
print(" Loading datasets...\n")

df_main = pd.read_csv("Diseases_and_Symptoms_dataset.csv")
symptom_cols = [c for c in df_main.columns if c != "diseases"]
print(f"  Diseases_and_Symptoms_dataset.csv -> {len(df_main):,} baris, {len(df_main.columns)} kolom")
print(f"     -> {df_main['diseases'].nunique()} penyakit unik, {len(symptom_cols)} gejala\n")

df_desc = pd.read_csv("description.csv")
print(f"  description.csv -> {len(df_desc)} penyakit dengan deskripsi")

df_meds = pd.read_csv("medications.csv")
print(f"  medications.csv -> {len(df_meds)} penyakit dengan obat")

df_prec = pd.read_csv("precautions.csv")
print(f"  precautions.csv -> {len(df_prec)} penyakit dengan pencegahan")

df_workout = pd.read_csv("workout.csv")
print(f"  workout.csv -> {len(df_workout)} penyakit dengan olahraga")

df_diet = pd.read_csv("diets.csv")
print(f"  diets.csv -> {len(df_diet)} penyakit dengan diet")

# Helper to parse list-strings
def parse_list_string(s):
    """Parse string list seperti "['item1', 'item2']" """
    try:
        return ast.literal_eval(s)
    except:
        return [x.strip().strip("'\"") for x in str(s).strip("[]").split(",")]

# Preview dataset
print("\n" + "="*60)
print("PREVIEW DATA")
print("="*60)

print("\nDiseases & Symptoms (5 kolom pertama):")
display(df_main.iloc[:5, :6])

print("\nDescription:")
display(df_desc.head())

print("\nMedications:")
display(df_meds.head())

 Loading datasets...

  Diseases_and_Symptoms_dataset.csv -> 96,088 baris, 231 kolom
     -> 100 penyakit unik, 230 gejala

  description.csv -> 100 penyakit dengan deskripsi
  medications.csv -> 100 penyakit dengan obat
  precautions.csv -> 100 penyakit dengan pencegahan
  workout.csv -> 100 penyakit dengan olahraga
  diets.csv -> 100 penyakit dengan diet

PREVIEW DATA

Diseases & Symptoms (5 kolom pertama):


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain
0,panic disorder,1,0,1,1,0
1,panic disorder,0,0,1,1,0
2,panic disorder,1,1,1,1,0
3,panic disorder,1,0,0,1,0
4,panic disorder,1,1,0,0,0



Description:


,Disease,Description
0,Panic disorder,Panic disorder is a mental health condition ma...
1,Vaginitis,Vaginitis is inflammation of the vaginal tissu...
2,Problem during pregnancy,Problems during pregnancy refer to medical com...
3,Acute pancreatitis,Acute pancreatitis is a sudden inflammation of...
4,Asthma,Asthma is a chronic inflammatory disease of th...



Medications:


,Disease,Medication
0,Panic disorder,"['SSRIs (e.g., Sertraline, Fluoxetine)', 'Benz..."
1,Vaginitis,"['Metronidazole', 'Clindamycin', 'Fluconazole'..."
2,Problem during pregnancy,"['Prenatal vitamins', 'Iron supplements', 'Ant..."
3,Acute pancreatitis,"['IV fluids', 'Pain relievers (e.g., Morphine)..."
4,Asthma,"['Inhaled corticosteroids (e.g., Fluticasone)'..."


---
##  Cell 5 — Setup Neo4j Constraints
Membuat uniqueness constraints untuk setiap jenis node agar tidak ada duplikat.

In [ ]:
# Install Neo4j di VM Colab
!wget -O - https://debian.neo4j.com/neotechnology.gpg.key | sudo apt-key add -
!echo 'deb https://debian.neo4j.com stable latest' | sudo tee -a /etc/apt/sources.list.d/neo4j.list
!sudo apt-get update
!sudo apt-get install neo4j -y

# Mulai service Neo4j
!sudo service neo4j start

--2026-06-23 14:25:47--  https://debian.neo4j.com/neotechnology.gpg.key
Resolving debian.neo4j.com (debian.neo4j.com)... 99.84.215.42, 99.84.215.77, 99.84.215.124, ...
Connecting to debian.neo4j.com (debian.neo4j.com)|99.84.215.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3905 (3.8K) [application/pgp-keys]
Saving to: ‘STDOUT’

-                   100%[===================>]   3.81K  --.-KB/s    in 0s      

2026-06-23 14:25:47 (3.15 GB/s) - written to stdout [3905/3905]

OK
deb https://debian.neo4j.com stable latest
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://debian.neo4j.com stable InRelease [44.3 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease

In [ ]:
NEO4J_URI      = "bolt://127.0.0.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "neo4j" # password default Neo4j saat pertama kali di-install

In [ ]:
from neo4j import GraphDatabase

# Connect to Neo4j
print("Connecting ke Neo4j...")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Coba paksa reset password jika expired
try:
    print("Memeriksa status password...")
    with driver.session(database="system") as session:
        # Jalankan query test untuk memicu pengecekan status password
        session.run("SHOW CURRENT USER")
except Exception as e:
    if "CredentialsExpired" in str(e) or "must be changed" in str(e):
        print("Mendeteksi password expired / harus diubah. Mencoba memperbarui password...")
        try:
            with driver.session(database="system") as session:
                # Skenario 1: Jika password default 'neo4j', ubah ke 'emriq123'
                if NEO4J_PASSWORD == "neo4j":
                    session.run("ALTER CURRENT USER SET PASSWORD FROM 'neo4j' TO 'emriq123'")
                    print("Password default 'neo4j' berhasil diubah ke 'emriq123'!")
                    NEO4J_PASSWORD = "emriq123"
                else:
                    # Skenario 2: Jika password sudah 'emriq123', coba setel ulang ke dirinya sendiri
                    try:
                        session.run("ALTER CURRENT USER SET PASSWORD FROM $pwd TO $pwd", pwd=NEO4J_PASSWORD)
                        print(f"Password berhasil di-refresh ke: {NEO4J_PASSWORD}")
                    except Exception as same_err:
                        # Skenario 3: Jika Neo4j melarang menggunakan password yang sama dengan password lama
                        new_pwd = NEO4J_PASSWORD + "4"
                        session.run("ALTER CURRENT USER SET PASSWORD FROM $curr TO $new", curr=NEO4J_PASSWORD, new=new_pwd)
                        print(f"Password diperbarui ke password baru: {new_pwd}")
                        print(f"PENTING: Silakan ubah password di Cell 2 Anda menjadi '{new_pwd}'")
                        NEO4J_PASSWORD = new_pwd

            # Hubungkan kembali dengan password yang baru diperbarui
            driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
            print("Koneksi berhasil terjalin kembali dengan password baru!")
        except Exception as change_err:
            print(f"Gagal merubah password secara otomatis: {change_err}")
    else:
        print(f"Koneksi error: {e}")

# Set up constraints
constraints = [
    ("Disease",    "CREATE CONSTRAINT disease_name IF NOT EXISTS FOR (d:Disease) REQUIRE d.name IS UNIQUE"),
    ("Symptom",    "CREATE CONSTRAINT symptom_name IF NOT EXISTS FOR (s:Symptom) REQUIRE s.name IS UNIQUE"),
    ("Medication", "CREATE CONSTRAINT med_name IF NOT EXISTS FOR (m:Medication) REQUIRE m.name IS UNIQUE"),
    ("Precaution", "CREATE CONSTRAINT prec_name IF NOT EXISTS FOR (p:Precaution) REQUIRE p.name IS UNIQUE"),
    ("Workout",    "CREATE CONSTRAINT workout_name IF NOT EXISTS FOR (w:Workout) REQUIRE w.name IS UNIQUE"),
    ("Diet",       "CREATE CONSTRAINT diet_name IF NOT EXISTS FOR (dt:Diet) REQUIRE dt.name IS UNIQUE"),
]

print("Membuat uniqueness constraints...")
with driver.session() as session:
    for label, query in constraints:
        try:
            session.run(query)
            print(f"  Constraint :{label} berhasil dibuat")
        except Exception as e:
            print(f"  :{label} — {e}")

print("\nSetup constraints selesai!")

Connecting ke Neo4j...
Memeriksa status password...
Mendeteksi password expired / harus diubah. Mencoba memperbarui password...
Password default 'neo4j' berhasil diubah ke 'emriq123'!
Koneksi berhasil terjalin kembali dengan password baru!
Membuat uniqueness constraints...
  Constraint :Disease berhasil dibuat
  Constraint :Symptom berhasil dibuat
  Constraint :Medication berhasil dibuat
  Constraint :Precaution berhasil dibuat
  Constraint :Workout berhasil dibuat
  Constraint :Diet berhasil dibuat

Setup constraints selesai!


---
##  Cell 6 — Ingest Data ke Neo4j
Memasukkan semua data dari 6 CSV ke dalam Neo4j sebagai nodes dan relationships.

**Entitas:** `Disease`, `Symptom`, `Medication`, `Precaution`, `Workout`, `Diet`

**Relasi:** `HAS_SYMPTOM`, `TREATED_WITH`, `HAS_PRECAUTION`, `RECOMMENDED_WORKOUT`, `RECOMMENDED_DIET`

In [ ]:
# Ingest diseases and symptoms
print("=" * 60)
print("STEP 1: Loading Diseases & Symptoms")
print("=" * 60)

disease_symptoms = {}
for disease, group in df_main.groupby("diseases"):
    symptom_freq = group[symptom_cols].mean()
    active_symptoms = symptom_freq[symptom_freq > 0.1].index.tolist()
    disease_symptoms[disease] = active_symptoms

print(f"  -> {len(disease_symptoms)} penyakit unik")
print(f"  -> {len(symptom_cols)} gejala tersedia\n")

def ingest_diseases_and_symptoms(tx, disease_name, symptoms):
    tx.run("MERGE (d:Disease {name: $name})", name=disease_name)
    for symptom in symptoms:
        tx.run("""
            MERGE (s:Symptom {name: $symptom})
            WITH s
            MATCH (d:Disease {name: $disease})
            MERGE (d)-[:HAS_SYMPTOM]->(s)
        """, symptom=symptom, disease=disease_name)

with driver.session() as session:
    total = len(disease_symptoms)
    for i, (disease, symptoms) in enumerate(disease_symptoms.items(), 1):
        session.execute_write(ingest_diseases_and_symptoms, disease, symptoms)
        if i % 20 == 0 or i == total:
            print(f"  [{i:>3}/{total}] Progress: {i/total*100:.0f}%")

print(f"Step 1 done — {total} diseases + symptoms di-ingest\n")

# Ingest descriptions
print("=" * 60)
print("STEP 2: Loading Disease Descriptions")
print("=" * 60)

def ingest_description(tx, disease, description):
    tx.run("""
        MERGE (d:Disease {name: $name})
        SET d.description = $desc
    """, name=disease, desc=description)

with driver.session() as session:
    matched = 0
    for _, row in df_desc.iterrows():
        disease_name = str(row["Disease"]).strip().lower()
        session.execute_write(ingest_description, disease_name, str(row["Description"]))
        matched += 1
    print(f"  {matched} deskripsi di-update\n")

# Ingest medications
print("=" * 60)
print("STEP 3: Loading Medications")
print("=" * 60)

def ingest_medications(tx, disease, medications):
    for med in medications:
        med = med.strip()
        if not med:
            continue
        tx.run("""
            MERGE (m:Medication {name: $med})
            WITH m
            MERGE (d:Disease {name: $disease})
            MERGE (d)-[:TREATED_WITH]->(m)
        """, med=med, disease=disease)

with driver.session() as session:
    for _, row in df_meds.iterrows():
        disease_name = str(row["Disease"]).strip().lower()
        meds = parse_list_string(row["Medication"])
        session.execute_write(ingest_medications, disease_name, meds)
    print(f"  {len(df_meds)} penyakit + obat di-ingest\n")

# Ingest precautions
print("=" * 60)
print("STEP 4: Loading Precautions")
print("=" * 60)

def ingest_precautions(tx, disease, precautions):
    for prec in precautions:
        prec = prec.strip()
        if not prec:
            continue
        tx.run("""
            MERGE (p:Precaution {name: $prec})
            WITH p
            MERGE (d:Disease {name: $disease})
            MERGE (d)-[:HAS_PRECAUTION]->(p)
        """, prec=prec, disease=disease)

with driver.session() as session:
    for _, row in df_prec.iterrows():
        disease_name = str(row["Disease"]).strip().lower()
        precs = [str(row[f"Precaution_{i}"]).strip()
                 for i in range(1, 5)
                 if pd.notna(row.get(f"Precaution_{i}"))]
        session.execute_write(ingest_precautions, disease_name, precs)
    print(f"  {len(df_prec)} penyakit + precautions di-ingest\n")

# Ingest workouts
print("=" * 60)
print("STEP 5: Loading Workouts")
print("=" * 60)

def ingest_workouts(tx, disease, workouts):
    for w in workouts:
        w_name = w.split(":")[0].strip() if ":" in w else w.strip()
        if not w_name:
            continue
        tx.run("""
            MERGE (w:Workout {name: $workout})
            WITH w
            MERGE (d:Disease {name: $disease})
            MERGE (d)-[:RECOMMENDED_WORKOUT]->(w)
        """, workout=w_name, disease=disease)

with driver.session() as session:
    for _, row in df_workout.iterrows():
        disease_name = str(row["Disease"]).strip().lower()
        workouts = parse_list_string(row["Workouts"])
        session.execute_write(ingest_workouts, disease_name, workouts)
    print(f"  {len(df_workout)} penyakit + workouts di-ingest\n")

# Ingest diets
print("=" * 60)
print("STEP 6: Loading Diets")
print("=" * 60)

def ingest_diets(tx, disease, diets):
    for d_item in diets:
        d_item = d_item.strip()
        if not d_item:
            continue
        tx.run("""
            MERGE (dt:Diet {name: $diet})
            WITH dt
            MERGE (d:Disease {name: $disease})
            MERGE (d)-[:RECOMMENDED_DIET]->(dt)
        """, diet=d_item, disease=disease)

with driver.session() as session:
    for _, row in df_diet.iterrows():
        disease_name = str(row["Disease"]).strip().lower()
        diets = parse_list_string(row["Diet"])
        session.execute_write(ingest_diets, disease_name, diets)
    print(f"  {len(df_diet)} penyakit + diets di-ingest\n")

print("=" * 60)
print("SEMUA DATA BERHASIL DI-INGEST KE NEO4J!")
print("=" * 60)

STEP 1: Loading Diseases & Symptoms
  -> 100 penyakit unik
  -> 230 gejala tersedia

  [ 20/100] Progress: 20%
  [ 40/100] Progress: 40%
  [ 60/100] Progress: 60%
  [ 80/100] Progress: 80%
  [100/100] Progress: 100%
Step 1 done — 100 diseases + symptoms di-ingest

STEP 2: Loading Disease Descriptions
  100 deskripsi di-update

STEP 3: Loading Medications
  100 penyakit + obat di-ingest

STEP 4: Loading Precautions
  100 penyakit + precautions di-ingest

STEP 5: Loading Workouts
  100 penyakit + workouts di-ingest

STEP 6: Loading Diets
  100 penyakit + diets di-ingest

SEMUA DATA BERHASIL DI-INGEST KE NEO4J!


---
## Cell 7 — Verifikasi Graph
Menghitung total nodes dan relationships yang berhasil di-ingest.

In [ ]:
print(" Statistik Graph Database\n")
print("=" * 55)
print("  NODES")
print("=" * 55)

with driver.session() as session:
    result = session.run("""
        MATCH (n)
        RETURN labels(n)[0] AS NodeType, COUNT(n) AS Total
        ORDER BY Total DESC
    """)
    total_nodes = 0
    for r in result:
        print(f"  {r['NodeType']:>15}: {r['Total']:>5} nodes")
        total_nodes += r['Total']

    print(f"  {'─'*30}")
    print(f"  {'TOTAL':>15}: {total_nodes:>5} nodes")

    print(f"\n{'=' * 55}")
    print("  RELATIONSHIPS")
    print("=" * 55)

    result2 = session.run("""
        MATCH ()-[r]->()
        RETURN TYPE(r) AS RelType, COUNT(r) AS Total
        ORDER BY Total DESC
    """)
    total_rels = 0
    for r in result2:
        print(f"  {r['RelType']:>25}: {r['Total']:>5} relasi")
        total_rels += r['Total']

    print(f"  {'─'*36}")
    print(f"  {'TOTAL':>25}: {total_rels:>5} relasi")

print(f"\nGraph memiliki {total_nodes} nodes dan {total_rels} relationships")
print(f"   Bisa dilihat di Neo4j Browser: http://localhost:7474")

 Statistik Graph Database

  NODES
       Medication:   402 nodes
       Precaution:   336 nodes
             Diet:   309 nodes
          Workout:   234 nodes
          Symptom:   230 nodes
          Disease:   101 nodes
  ──────────────────────────────
            TOTAL:  1612 nodes

  RELATIONSHIPS
                HAS_SYMPTOM:  1139 relasi
               TREATED_WITH:   500 relasi
           RECOMMENDED_DIET:   496 relasi
             HAS_PRECAUTION:   400 relasi
        RECOMMENDED_WORKOUT:   400 relasi
  ────────────────────────────────────
                      TOTAL:  2935 relasi

Graph memiliki 1612 nodes dan 2935 relationships
   Bisa dilihat di Neo4j Browser: http://localhost:7474


---
## Cell 8 — LLM Text-to-Cypher (Tier 1)
LLM menerjemahkan pertanyaan **bahasa alami** menjadi **query Cypher**, lalu mengeksekusinya di Neo4j.

**Flow:** `User Question -> LLM -> Cypher Query -> Neo4j -> Results`

In [ ]:
# Ambil schema graph dari Neo4j untuk diberikan ke LLM
def get_graph_schema():
    """Get graph schema details."""
    schema_parts = []

    with driver.session() as session:
        # Node labels dan properties
        node_result = session.run("""
            CALL db.schema.nodeTypeProperties()
            YIELD nodeLabels, propertyName, propertyTypes
            RETURN nodeLabels, collect(propertyName) AS properties
        """)
        schema_parts.append("Node Labels & Properties:")
        for r in node_result:
            labels = r["nodeLabels"]
            props = r["properties"]
            schema_parts.append(f"  :{labels} -> {props}")

        # Relationship types
        rel_result = session.run("""
            CALL db.schema.relTypeProperties()
            YIELD relType
            RETURN DISTINCT relType
        """)
        schema_parts.append("\nRelationship Types:")
        for r in rel_result:
            schema_parts.append(f"  {r['relType']}")

        # Relationship patterns
        pattern_result = session.run("""
            MATCH (a)-[r]->(b)
            RETURN DISTINCT labels(a)[0] AS from_label, type(r) AS rel_type, labels(b)[0] AS to_label
        """)
        schema_parts.append("\nRelationship Patterns:")
        for r in pattern_result:
            schema_parts.append(f"  (:{r['from_label']})-[:{r['rel_type']}]->(:{r['to_label']})")

    return "\n".join(schema_parts)

graph_schema = get_graph_schema()
print("Graph Schema:\n")
print(graph_schema)

Graph Schema:

Node Labels & Properties:
  :['Symptom'] -> ['name']
  :['Medication'] -> ['name']
  :['Precaution'] -> ['name']
  :['Workout'] -> ['name']
  :['Diet'] -> ['name']
  :['Disease'] -> ['name', 'description']

Relationship Types:
  :`HAS_SYMPTOM`
  :`TREATED_WITH`
  :`HAS_PRECAUTION`
  :`RECOMMENDED_WORKOUT`
  :`RECOMMENDED_DIET`

Relationship Patterns:
  (:Disease)-[:HAS_SYMPTOM]->(:Symptom)
  (:Disease)-[:TREATED_WITH]->(:Medication)
  (:Disease)-[:HAS_PRECAUTION]->(:Precaution)
  (:Disease)-[:RECOMMENDED_WORKOUT]->(:Workout)
  (:Disease)-[:RECOMMENDED_DIET]->(:Diet)


In [ ]:
# Text-to-Cypher Function
TEXT_TO_CYPHER_PROMPT = """You are a Neo4j Cypher expert. Generate a READ-ONLY Cypher query based on the user's question and the graph schema below.

GRAPH SCHEMA:
{schema}

RULES:
1. Output ONLY the Cypher query, nothing else. No explanations, no markdown formatting.
2. Use only MATCH and RETURN (read-only). Never use CREATE, DELETE, SET, MERGE, or DETACH.
3. Use toLower() for case-insensitive string matching.
4. Always LIMIT results (max 10) unless the user asks for all.
5. For symptom-based questions, match symptoms and count overlaps to rank diseases.
6. Node property for names is always .name (lowercase).
7. Disease names in the database are lowercase.

EXAMPLES:
User: "Penyakit apa yang punya gejala fever dan headache?"
Cypher: MATCH (d:Disease)-[:HAS_SYMPTOM]->(s:Symptom) WHERE toLower(s.name) IN ['fever', 'headache'] WITH d, collect(s.name) AS matched, count(s) AS cnt RETURN d.name AS disease, matched AS matched_symptoms, cnt AS match_count ORDER BY cnt DESC LIMIT 10

User: "Obat untuk malaria?"
Cypher: MATCH (d:Disease)-[:TREATED_WITH]->(m:Medication) WHERE toLower(d.name) = 'malaria' RETURN d.name AS disease, collect(m.name) AS medications

User: "Berapa jumlah gejala diabetes?"
Cypher: MATCH (d:Disease)-[:HAS_SYMPTOM]->(s:Symptom) WHERE toLower(d.name) = 'diabetes' RETURN d.name AS disease, count(s) AS total_symptoms, collect(s.name) AS symptoms

User: "Rekomendasi diet dan olahraga untuk asthma?"
Cypher: MATCH (d:Disease) WHERE toLower(d.name) = 'asthma' OPTIONAL MATCH (d)-[:RECOMMENDED_DIET]->(dt:Diet) OPTIONAL MATCH (d)-[:RECOMMENDED_WORKOUT]->(w:Workout) RETURN d.name AS disease, collect(DISTINCT dt.name) AS diets, collect(DISTINCT w.name) AS workouts

NOW GENERATE CYPHER FOR THIS QUESTION:
{question}"""

def text_to_cypher(question):
    """Translate natural language question to Cypher and execute."""
    print(f"\n{'='*60}")
    print(f" Pertanyaan: {question}")
    print(f"{'='*60}")

    # Step 1: LLM generates Cypher
    prompt = TEXT_TO_CYPHER_PROMPT.format(schema=graph_schema, question=question)
    cypher_query = tanya(prompt).strip()

    # Clean up: remove markdown code fences if present
    cypher_query = cypher_query.replace("```cypher", "").replace("```", "").strip()

    print(f"\n Generated Cypher:\n   {cypher_query}")

    # Step 2: Execute on Neo4j
    try:
        with driver.session() as session:
            result = session.run(cypher_query)
            records = [record.data() for record in result]

        print(f"\n Results ({len(records)} rows):")
        for i, record in enumerate(records, 1):
            print(f"   [{i}] {record}")

        return {"query": cypher_query, "results": records}

    except Exception as e:
        print(f"\nError executing Cypher: {e}")
        return {"query": cypher_query, "error": str(e)}

# Demo Text-to-Cypher
print("DEMO: LLM Text-to-Cypher\n")

demo_questions = [
    "Penyakit apa saja yang memiliki gejala fever dan headache?",
    "Obat apa yang direkomendasikan untuk common cold?",
    "Berikan rekomendasi diet dan olahraga untuk diabetes",
]

for q in demo_questions:
    text_to_cypher(q)
    print()

DEMO: LLM Text-to-Cypher


 Pertanyaan: Penyakit apa saja yang memiliki gejala fever dan headache?

 Generated Cypher:
   MATCH (d:Disease)-[:HAS_SYMPTOM]->(s:Symptom) WHERE toLower(s.name) IN ['fever', 'headache'] WITH d, collect(s.name) AS matched, count(s) AS cnt RETURN d.name AS disease, matched AS matched_symptoms, cnt AS match_count ORDER BY cnt DESC LIMIT 10

 Results (10 rows):
   [1] {'disease': 'common cold', 'matched_symptoms': ['fever', 'headache'], 'match_count': 2}
   [2] {'disease': 'noninfectious gastroenteritis', 'matched_symptoms': ['fever', 'headache'], 'match_count': 2}
   [3] {'disease': 'strep throat', 'matched_symptoms': ['fever', 'headache'], 'match_count': 2}
   [4] {'disease': 'acute bronchitis', 'matched_symptoms': ['fever', 'headache'], 'match_count': 2}
   [5] {'disease': 'infectious gastroenteritis', 'matched_symptoms': ['fever', 'headache'], 'match_count': 2}
   [6] {'disease': 'nose disorder', 'matched_symptoms': ['fever', 'headache'], 'match_count': 2}


---
##  Cell 9 — Graph Analytics: PageRank (Tier 1)
**PageRank** mengidentifikasi node yang paling "penting" dalam graph berdasarkan jumlah dan kualitas koneksi.

Node dengan PageRank tinggi = node yang paling banyak terhubung dan paling berpengaruh dalam jaringan medis.

In [ ]:
# 1. Install package Graph Data Science
!sudo apt-get install neo4j-graph-data-science -y

# 2. Berikan izin akses prosedur GDS di file konfigurasi neo4j.conf
!echo "dbms.security.procedures.unrestricted=gds.*" | sudo tee -a /etc/neo4j/neo4j.conf
!echo "dbms.security.procedures.allowlist=gds.*" | sudo tee -a /etc/neo4j/neo4j.conf

# 3. Restart service Neo4j agar plugin termuat
!sudo service neo4j restart

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package neo4j-graph-data-science
dbms.security.procedures.unrestricted=gds.*
dbms.security.procedures.allowlist=gds.*
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Stopping Neo4j............ stopped.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Directories in use:
home:         /var/lib/neo4j
config:       /etc/neo

In [ ]:
%%bash
# 1. Coba salin dari folder 'products' bawaan Neo4j (jika ada)
if [ -d "/usr/share/neo4j/products" ]; then
    sudo cp /usr/share/neo4j/products/neo4j-graph-data-science-*.jar /var/lib/neo4j/plugins/
elif [ -d "/var/lib/neo4j/products" ]; then
    sudo cp /var/lib/neo4j/products/neo4j-graph-data-science-*.jar /var/lib/neo4j/plugins/
fi

# 2. Jika tidak ada di folder products, download langsung dari GitHub Releases resmi GDS
if [ ! -f /var/lib/neo4j/plugins/neo4j-graph-data-science-*.jar ]; then
    echo "File GDS tidak ditemukan di folder products. Mengunduh GDS versi 2.6.8 dari GitHub..."
    sudo wget -P /var/lib/neo4j/plugins/ https://github.com/neo4j/graph-data-science/releases/download/2.6.8/neo4j-graph-data-science-2.6.8.jar
fi

# 3. Atur permission file agar dapat dibaca oleh Neo4j
sudo chmod 755 /var/lib/neo4j/plugins/neo4j-graph-data-science-*.jar

# 4. Tambahkan konfigurasi izin prosedur ke neo4j.conf (jika belum ada)
if ! grep -q "dbms.security.procedures.unrestricted=gds.\*" /etc/neo4j/neo4j.conf; then
    echo "dbms.security.procedures.unrestricted=gds.*" | sudo tee -a /etc/neo4j/neo4j.conf
fi
if ! grep -q "dbms.security.procedures.allowlist=gds.\*" /etc/neo4j/neo4j.conf; then
    echo "dbms.security.procedures.allowlist=gds.*" | sudo tee -a /etc/neo4j/neo4j.conf
fi

# 5. Restart service Neo4j
echo "Mereset service Neo4j..."
sudo service neo4j restart

Mereset service Neo4j...
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Stopping Neo4j............ stopped.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Directories in use:
home:         /var/lib/neo4j
config:       /etc/neo4j
logs:         /var/log/neo4j
plugins:      /var/lib/neo4j/plugins
import:       /var/lib/neo4j/import
data:         /var/lib/neo4j/data
certificates: /var/lib/neo4j/certificates
licenses:     /var/lib/neo4j/

In [ ]:
# Project graph to GDS
def run_query(tx, query):
    result = tx.run(query)
    return [record.data() for record in result]

print(" Setting up GDS in-memory graph 'medigraph'...\n")

# Drop if exists
try:
    with driver.session() as session:
        session.execute_write(lambda tx: tx.run("CALL gds.graph.drop('medigraph', false)"))
        print("  ️ Dropped existing 'medigraph' projection")
except Exception:
    pass

# Project graph — semua 6 node types dan 5 relationship types
projection_query = """
CALL gds.graph.project(
    'medigraph',
    ['Disease', 'Symptom', 'Medication', 'Precaution', 'Workout', 'Diet'],
    {
        HAS_SYMPTOM: {orientation: 'UNDIRECTED'},
        TREATED_WITH: {orientation: 'UNDIRECTED'},
        HAS_PRECAUTION: {orientation: 'UNDIRECTED'},
        RECOMMENDED_WORKOUT: {orientation: 'UNDIRECTED'},
        RECOMMENDED_DIET: {orientation: 'UNDIRECTED'}
    }
)
YIELD graphName, nodeCount, relationshipCount
"""

with driver.session() as session:
    result = session.execute_write(run_query, projection_query)
    info = result[0]
    print(f"  Graph '{info['graphName']}' projected:")
    print(f"     -> {info['nodeCount']} nodes")
    print(f"     -> {info['relationshipCount']} relationships")

# Run PageRank
print(f"\n{'='*60}")
print(" PageRank — Top 15 Most Important Nodes")
print(f"{'='*60}\n")

pagerank_query = """
CALL gds.pageRank.stream('medigraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name,
       labels(gds.util.asNode(nodeId))[0] AS label,
       score
ORDER BY score DESC
LIMIT 15
"""

with driver.session() as session:
    results = session.execute_read(run_query, pagerank_query)
    print(f"  {'Rank':<6} {'Type':<12} {'Name':<35} {'Score':<10}")
    print(f"  {'─'*63}")
    for i, r in enumerate(results, 1):
        print(f"  {i:<6} {r['label']:<12} {r['name']:<35} {r['score']:.6f}")

print(f"\nInsight: Node dengan PageRank tertinggi adalah yang paling 'sentral'")
print(f"   dalam jaringan medis — paling banyak terhubung ke entitas lain.")

 Setting up GDS in-memory graph 'medigraph'...

  ️ Dropped existing 'medigraph' projection
  Graph 'medigraph' projected:
     -> 1612 nodes
     -> 5870 relationships

 PageRank — Top 15 Most Important Nodes

  Rank   Type         Name                                Score     
  ───────────────────────────────────────────────────────────────
  1      Diet         Hydration                           10.873854
  2      Disease      developmental disability            8.914020
  3      Disease      personality disorder                8.709253
  4      Disease      gum disease                         8.543823
  5      Disease      dental caries                       8.533848
  6      Disease      schizophrenia                       8.520114
  7      Disease      benign prostatic hyperplasia (bph)  8.314447
  8      Disease      obstructive sleep apnea (osa)       8.212862
  9      Disease      marijuana abuse                     8.190797
  10     Disease      vaginitis                   

---
## Cell 10 — Graph Analytics: Community Detection / Louvain (Tier 1)
**Louvain** mendeteksi komunitas/kluster dalam graph — kelompok penyakit yang saling terkait melalui gejala, obat, atau penanganan yang sama.

In [ ]:
print(f"{'='*60}")
print("Community Detection (Louvain) — Top 10 Communities")
print(f"{'='*60}\n")

louvain_query = """
CALL gds.louvain.stream('medigraph')
YIELD nodeId, communityId
WITH gds.util.asNode(nodeId) AS n, communityId,
     labels(gds.util.asNode(nodeId))[0] AS label
WHERE label = 'Disease'
RETURN communityId,
       collect(n.name)[0..5] AS sample_diseases,
       count(n) AS size
ORDER BY size DESC
LIMIT 10
"""

with driver.session() as session:
    results = session.execute_read(run_query, louvain_query)

    for r in results:
        diseases_str = ", ".join(r["sample_diseases"])
        print(f"   Community {r['communityId']} (Size: {r['size']} diseases)")
        print(f"     Contoh: {diseases_str}")
        print()

print("Insight: Penyakit dalam komunitas yang sama cenderung berbagi")
print("   gejala, obat, atau penanganan yang mirip.")

Community Detection (Louvain) — Top 10 Communities

   Community 128 (Size: 18 diseases)
     Contoh: arthritis of the hip, brachial neuritis, bursitis, carpal tunnel syndrome, chronic back pain

   Community 385 (Size: 13 diseases)
     Contoh: acute bronchiolitis, acute bronchitis, acute bronchospasm, acute sinusitis, asthma

   Community 512 (Size: 11 diseases)
     Contoh: actinic keratosis, allergy, contact dermatitis, drug reaction, eczema

   Community 608 (Size: 11 diseases)
     Contoh: acute kidney injury, acute pancreatitis, appendicitis, cholecystitis, diverticulitis

   Community 228 (Size: 10 diseases)
     Contoh: benign prostatic hyperplasia (bph), chronic constipation, cystitis, hemorrhoids, liver disease

   Community 239 (Size: 8 diseases)
     Contoh: esophagitis, heart attack, heart failure, hiatal hernia, hypertensive heart disease

   Community 287 (Size: 8 diseases)
     Contoh: angina, anxiety, depression, developmental disability, marijuana abuse

   Community

---
## Cell 11 — Graph Analytics: Shortest Path + Degree Centrality (Tier 1)
- **Shortest Path** — Mencari jalur terpendek antara dua penyakit melalui entitas yang menghubungkan
- **Degree Centrality** — Mengukur jumlah koneksi langsung setiap node

In [ ]:
print(f"{'='*60}")
print("Shortest Path Between Diseases")
print(f"{'='*60}\n")

# Ambil 2 penyakit random untuk demo
with driver.session() as session:
    sample = session.run("""
        MATCH (d:Disease)
        WITH d, rand() AS r
        ORDER BY r
        LIMIT 2
        RETURN collect(d.name) AS diseases
    """)
    sample_diseases = sample.single()["diseases"]

disease_a = sample_diseases[0]
disease_b = sample_diseases[1]
print(f"   Mencari jalur terpendek antara:")
print(f"     Disease A: {disease_a}")
print(f"     Disease B: {disease_b}\n")

shortest_path_query = """
MATCH (a:Disease {name: $disease_a}), (b:Disease {name: $disease_b})
MATCH path = shortestPath((a)-[*]-(b))
RETURN [node IN nodes(path) | coalesce(node.name, 'unknown')] AS path_nodes,
       [rel IN relationships(path) | type(rel)] AS path_rels,
       length(path) AS path_length
"""

with driver.session() as session:
    result = session.run(shortest_path_query, disease_a=disease_a, disease_b=disease_b)
    record = result.single()

    if record:
        print(f"  Path Length: {record['path_length']} hops\n")
        print(f"  Path:")
        nodes = record["path_nodes"]
        rels = record["path_rels"]
        path_str = nodes[0]
        for i, rel in enumerate(rels):
            path_str += f" --[{rel}]--> {nodes[i+1]}"
        print(f"     {path_str}")
    else:
        print("  Tidak ditemukan jalur antara kedua penyakit")

# Calculate degree centrality
print(f"\n{'='*60}")
print("Degree Centrality — Top 10 Most Connected Diseases")
print(f"{'='*60}\n")

degree_query = """
MATCH (d:Disease)
OPTIONAL MATCH (d)-[r]-()
WITH d.name AS disease, count(r) AS degree
ORDER BY degree DESC
LIMIT 10
RETURN disease, degree
"""

with driver.session() as session:
    results = session.execute_read(run_query, degree_query)
    print(f"  {'Rank':<6} {'Disease':<35} {'Connections':<12}")
    print(f"  {'─'*53}")
    for i, r in enumerate(results, 1):
        bar = '#' * min(int(r['degree'] / 2), 30)
        print(f"  {i:<6} {r['disease']:<35} {r['degree']:<5} {bar}")

Shortest Path Between Diseases

   Mencari jalur terpendek antara:
     Disease A: diverticulitis
     Disease B: skin polyp

  Path Length: 2 hops

  Path:
     diverticulitis --[RECOMMENDED_DIET]--> Hydration --[RECOMMENDED_DIET]--> skin polyp

Degree Centrality — Top 10 Most Connected Diseases

  Rank   Disease                             Connections 
  ─────────────────────────────────────────────────────
  1      cholecystitis                       30    ###############
  2      complex regional pain syndrome      30    ###############
  3      benign prostatic hyperplasia (bph)  30    ###############
  4      bursitis                            30    ###############
  5      acute pancreatitis                  30    ###############
  6      acute bronchitis                    30    ###############
  7      acute bronchiolitis                 30    ###############
  8      arthritis of the hip                30    ###############
  9      anxiety                             30    

---
## Cell 12 — Machine Learning on Graph: FastRP Embeddings + KNN (Tier 2)
1. **FastRP (Fast Random Projection)** — menghasilkan vector embeddings untuk setiap node berdasarkan struktur graph
2. **KNN (K-Nearest Neighbors)** — mencari penyakit yang paling mirip berdasarkan embeddings
3. **Node Similarity** — menghitung kesamaan antar node berdasarkan shared neighbors

In [ ]:
# FastRP Embeddings
print(f"{'='*60}")
print("FastRP Node Embeddings")
print(f"{'='*60}\n")

# Compute FastRP embeddings
fastrp_mutate = """
CALL gds.fastRP.mutate('medigraph', {
    embeddingDimension: 64,
    randomSeed: 42,
    mutateProperty: 'embedding'
})
YIELD nodePropertiesWritten
"""

with driver.session() as session:
    result = session.execute_write(run_query, fastrp_mutate)
    print(f"  FastRP computed: {result[0]['nodePropertiesWritten']} node embeddings generated")

# Write embeddings to database
fastrp_write = """
CALL gds.graph.nodeProperties.write('medigraph', ['embedding'])
YIELD propertiesWritten
"""

with driver.session() as session:
    result = session.execute_write(run_query, fastrp_write)
    print(f"  Embeddings saved to database: {result[0]['propertiesWritten']} properties written")

# KNN Similarity Search
print(f"\n{'='*60}")
print("KNN — Top Similar Disease Pairs")
print(f"{'='*60}\n")

knn_query = """
CALL gds.knn.stream('medigraph', {
    topK: 3,
    nodeProperties: ['embedding'],
    randomSeed: 42,
    concurrency: 1
})
YIELD node1, node2, similarity
WITH gds.util.asNode(node1) AS n1, gds.util.asNode(node2) AS n2, similarity
WHERE 'Disease' IN labels(n1) AND 'Disease' IN labels(n2)
RETURN n1.name AS disease_1, n2.name AS disease_2,
       round(similarity * 10000) / 10000 AS similarity_score
ORDER BY similarity_score DESC
LIMIT 15
"""

with driver.session() as session:
    results = session.execute_read(run_query, knn_query)

    print(f"  {'Rank':<6} {'Disease 1':<25} {'Disease 2':<25} {'Similarity':<10}")
    print(f"  {'─'*66}")
    for i, r in enumerate(results, 1):
        score = r['similarity_score']
        bar = '#' * int(score * 20)
        print(f"  {i:<6} {r['disease_1']:<25} {r['disease_2']:<25} {score:.4f} {bar}")

# Node Similarity — Shared Symptoms
print(f"\n{'='*60}")
print("Node Similarity — Diseases with Most Shared Symptoms")
print(f"{'='*60}\n")

similarity_query = """
MATCH (d1:Disease)-[:HAS_SYMPTOM]->(s:Symptom)<-[:HAS_SYMPTOM]-(d2:Disease)
WHERE id(d1) < id(d2)
WITH d1.name AS disease_1, d2.name AS disease_2,
     count(s) AS shared_symptoms,
     collect(s.name)[0..5] AS sample_symptoms
ORDER BY shared_symptoms DESC
LIMIT 10
RETURN disease_1, disease_2, shared_symptoms, sample_symptoms
"""

with driver.session() as session:
    results = session.execute_read(run_query, similarity_query)

    for i, r in enumerate(results, 1):
        symptoms_str = ", ".join(r["sample_symptoms"])
        print(f"  [{i}] {r['disease_1']} <-> {r['disease_2']}")
        print(f"      Shared: {r['shared_symptoms']} symptoms (e.g., {symptoms_str})")
        print()

print("Insight: Penyakit dengan similarity tinggi dan banyak shared symptoms")
print("   bisa saling misdiagnosis — penting untuk differential diagnosis.")

FastRP Node Embeddings

  FastRP computed: 1612 node embeddings generated
  Embeddings saved to database: 1612 properties written

KNN — Top Similar Disease Pairs

  Rank   Disease 1                 Disease 2                 Similarity
  ──────────────────────────────────────────────────────────────────

Node Similarity — Diseases with Most Shared Symptoms



  [1] infectious gastroenteritis <-> noninfectious gastroenteritis
      Shared: 11 symptoms (e.g., sharp abdominal pain, headache, vomiting, blood in stool, decreased appetite)

  [2] spinal stenosis <-> spondylosis
      Shared: 10 symptoms (e.g., loss of sensation, paresthesia, arm pain, hip pain, neck pain)

  [3] cholecystitis <-> gallstone
      Shared: 10 symptoms (e.g., burning abdominal pain, back pain, sharp abdominal pain, regurgitation.1, sharp chest pain)

  [4] acute bronchospasm <-> pneumonia
      Shared: 10 symptoms (e.g., shortness of breath, nasal congestion, fever, wheezing, sore throat)

  [5] skin pigmentation disorder <-> skin polyp
      Shared: 10 symptoms (e.g., acne or pimples, warts, skin growth, skin moles, skin swelling)

  [6] degenerative disc disease <-> spinal stenosis
      Shared: 10 symptoms (e.g., shoulder pain, lower body pain, hip pain, low back pain, loss of sensation)

  [7] marijuana abuse <-> personality disorder
      Shared: 10 symptoms (e.

---
## Cell 13 — LLM Graph Builder (Tier 3)
LLM mengekstraksi **entitas** dan **relasi** dari teks medis mentah (unstructured text), kemudian otomatis memasukkannya ke Neo4j.

**Flow:** `Raw Medical Text -> LLM Entity Extraction -> JSON -> Populate Neo4j Graph`

In [ ]:
# LLM Graph Builder functions
GRAPH_BUILDER_PROMPT = """You are a medical knowledge graph builder. Extract entities and relationships from the given medical text.

OUTPUT FORMAT (strict JSON, no markdown):
{{
  "entities": [
    {{"type": "Disease", "name": "disease name in lowercase"}},
    {{"type": "Symptom", "name": "symptom name in lowercase"}},
    {{"type": "Medication", "name": "medication name in lowercase"}},
    {{"type": "Precaution", "name": "precaution in lowercase"}},
    {{"type": "Diet", "name": "diet recommendation in lowercase"}},
    {{"type": "Workout", "name": "exercise recommendation in lowercase"}}
  ],
  "relationships": [
    {{"from": "entity name", "from_type": "Disease", "relation": "HAS_SYMPTOM", "to": "entity name", "to_type": "Symptom"}},
    {{"from": "entity name", "from_type": "Disease", "relation": "TREATED_WITH", "to": "entity name", "to_type": "Medication"}},
    {{"from": "entity name", "from_type": "Disease", "relation": "HAS_PRECAUTION", "to": "entity name", "to_type": "Precaution"}},
    {{"from": "entity name", "from_type": "Disease", "relation": "RECOMMENDED_DIET", "to": "entity name", "to_type": "Diet"}},
    {{"from": "entity name", "from_type": "Disease", "relation": "RECOMMENDED_WORKOUT", "to": "entity name", "to_type": "Workout"}}
  ]
}}

RULES:
1. Extract ALL entities mentioned in the text.
2. Only use the entity types and relationship types listed above.
3. All names should be lowercase.
4. Output valid JSON only, no explanation.

MEDICAL TEXT:
{text}"""

def extract_entities_from_text(raw_text):
    """Extract entities and relationships from medical text using LLM."""
    print(f"\n{'='*60}")
    print(f"LLM Graph Builder — Entity Extraction")
    print(f"{'='*60}")
    print(f"\n Input Text:\n   {raw_text[:200]}{'...' if len(raw_text) > 200 else ''}\n")

    prompt = GRAPH_BUILDER_PROMPT.format(text=raw_text)
    response = tanya(prompt)

    # Parse JSON from response
    try:
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            extracted = json.loads(json_match.group())
        else:
            extracted = json.loads(response)

        entities = extracted.get("entities", [])
        relationships = extracted.get("relationships", [])

        print(f"  Extracted {len(entities)} entities:")
        for e in entities:
            print(f"     • [{e['type']}] {e['name']}")

        print(f"\n  Extracted {len(relationships)} relationships:")
        for r in relationships:
            print(f"     • ({r['from']}) -[{r['relation']}]-> ({r['to']})")

        return extracted

    except (json.JSONDecodeError, KeyError) as e:
        print(f"  Error parsing LLM response: {e}")
        print(f"  Raw response: {response[:300]}")
        return {"entities": [], "relationships": []}


def populate_graph_from_extraction(extracted):
    """Insert extracted entities and relationships into Neo4j."""
    print(f"\n{'='*60}")
    print(f" Populating Neo4j from Extracted Data")
    print(f"{'='*60}\n")

    entities = extracted.get("entities", [])
    relationships = extracted.get("relationships", [])

    valid_types = {"Disease", "Symptom", "Medication", "Precaution", "Workout", "Diet"}
    valid_rels = {"HAS_SYMPTOM", "TREATED_WITH", "HAS_PRECAUTION", "RECOMMENDED_WORKOUT", "RECOMMENDED_DIET"}

    with driver.session() as session:
        # Insert entities
        for entity in entities:
            etype = entity["type"]
            ename = entity["name"].strip().lower()
            if etype in valid_types:
                session.run(f"MERGE (n:{etype} {{name: $name}})", name=ename)
                print(f"  MERGE :{etype} -> {ename}")

        # Insert relationships
        for rel in relationships:
            from_type = rel["from_type"]
            to_type = rel["to_type"]
            relation = rel["relation"]
            from_name = rel["from"].strip().lower()
            to_name = rel["to"].strip().lower()

            if relation in valid_rels and from_type in valid_types and to_type in valid_types:
                query = f"""
                    MATCH (a:{from_type} {{name: $from_name}})
                    MATCH (b:{to_type} {{name: $to_name}})
                    MERGE (a)-[:{relation}]->(b)
                """
                session.run(query, from_name=from_name, to_name=to_name)
                print(f"  ({from_name}) -[{relation}]-> ({to_name})")

    print(f"\nGraph berhasil di-update dengan data baru!")


# Demo Graph Builder
print("DEMO: LLM Graph Builder\n")

demo_texts = [
    """
    Dengue fever is a mosquito-borne tropical disease caused by the dengue virus.
    Symptoms include high fever, severe headache, pain behind the eyes, joint and muscle pain,
    fatigue, nausea, vomiting, and skin rash. Treatment involves acetaminophen for pain relief,
    oral rehydration salts, and IV fluids in severe cases. Patients should rest, drink plenty
    of fluids, avoid aspirin, and use mosquito repellent. A diet rich in papaya leaf extract,
    coconut water, and vitamin C foods is recommended. Light stretching and walking are
    suggested as recovery exercises.
    """,
    """
    Migraine is a neurological condition characterized by intense, recurring headaches often
    on one side of the head. Common symptoms include throbbing pain, sensitivity to light and
    sound, nausea, and visual disturbances called aura. Medications include triptans like
    sumatriptan, NSAIDs such as ibuprofen, and preventive drugs like topiramate. Precautions
    include avoiding triggers like stress and bright lights, maintaining regular sleep, and
    keeping a headache diary. Recommended foods include magnesium-rich foods, omega-3 fatty
    acids, and ginger tea. Yoga and aerobic exercise can help reduce frequency.
    """
]

for text in demo_texts:
    extracted = extract_entities_from_text(text)
    populate_graph_from_extraction(extracted)
    print("\n" + "─" * 60)

DEMO: LLM Graph Builder


LLM Graph Builder — Entity Extraction

 Input Text:
   
    Dengue fever is a mosquito-borne tropical disease caused by the dengue virus.
    Symptoms include high fever, severe headache, pain behind the eyes, joint and muscle pain,
    fatigue, nausea, v...

  Extracted 19 entities:
     • [Disease] dengue fever
     • [Symptom] high fever
     • [Symptom] severe headache
     • [Symptom] pain behind the eyes
     • [Symptom] joint and muscle pain
     • [Symptom] fatigue
     • [Symptom] nausea
     • [Symptom] vomiting
     • [Symptom] skin rash
     • [Medication] acetaminophen
     • [Medication] oral rehydration salts
     • [Precaution] rest
     • [Precaution] avoid aspirin
     • [Precaution] use mosquito repellent
     • [Diet] papaya leaf extract
     • [Diet] coconut water
     • [Diet] vitamin c foods
     • [Workout] light stretching
     • [Workout] walking

  Extracted 18 relationships:
     • (dengue fever) -[HAS_SYMPTOM]-> (high fever)
     •

---
## Cell 14 — Graph-Augmented RAG (Tier 4)
**RAG (Retrieval-Augmented Generation)** yang diperkuat oleh graph:

1. **Retrieve** — Ambil konteks relevan dari Neo4j (via Cypher query yang di-generate LLM)
2. **Augment** — Gabungkan konteks graph ke dalam prompt LLM
3. **Generate** — LLM menghasilkan jawaban komprehensif berdasarkan data graph + knowledge internal

Ini adalah **tier tertinggi** yang menggabungkan semua komponen menjadi satu pipeline yang powerful.

In [ ]:
# Graph RAG pipeline
def retrieve_graph_context(question):
    """
    Step 1 (Retrieve): Generate Cypher via LLM, execute, and return structured context.
    Uses multiple queries to gather comprehensive context.
    """
    context_parts = []

    cypher_prompt = TEXT_TO_CYPHER_PROMPT.format(schema=graph_schema, question=question)
    main_cypher = tanya(cypher_prompt).strip().replace("```cypher", "").replace("```", "").strip()

    try:
        with driver.session() as session:
            result = session.run(main_cypher)
            records = [record.data() for record in result]
            if records:
                context_parts.append(f"Main Query Results:\n{json.dumps(records, indent=2, default=str)}")
    except Exception as e:
        context_parts.append(f"Main query error: {e}")

    enrichment_prompt = f"""From this medical question, extract any disease names or symptom names mentioned.
Output a JSON object: {{"diseases": ["name1"], "symptoms": ["name1"]}}
Only output JSON, nothing else.

Question: {question}"""

    try:
        entities_response = tanya(enrichment_prompt).strip()
        json_match = re.search(r'\{.*\}', entities_response, re.DOTALL)
        if json_match:
            entities = json.loads(json_match.group())

            # Get full context for mentioned diseases
            for disease in entities.get("diseases", []):
                with driver.session() as session:
                    full_info = session.run("""
                        MATCH (d:Disease)
                        WHERE toLower(d.name) CONTAINS toLower($name)
                        OPTIONAL MATCH (d)-[:HAS_SYMPTOM]->(s:Symptom)
                        OPTIONAL MATCH (d)-[:TREATED_WITH]->(m:Medication)
                        OPTIONAL MATCH (d)-[:HAS_PRECAUTION]->(p:Precaution)
                        OPTIONAL MATCH (d)-[:RECOMMENDED_DIET]->(dt:Diet)
                        OPTIONAL MATCH (d)-[:RECOMMENDED_WORKOUT]->(w:Workout)
                        RETURN d.name AS disease,
                               d.description AS description,
                               collect(DISTINCT s.name) AS symptoms,
                               collect(DISTINCT m.name) AS medications,
                               collect(DISTINCT p.name) AS precautions,
                               collect(DISTINCT dt.name) AS diets,
                               collect(DISTINCT w.name) AS workouts
                        LIMIT 3
                    """, name=disease)
                    for r in full_info:
                        context_parts.append(f"\nFull info for {r['disease']}:\n{json.dumps(r.data(), indent=2, default=str)}")

            # Get diseases matching mentioned symptoms
            symptoms = entities.get("symptoms", [])
            if symptoms:
                with driver.session() as session:
                    symptom_match = session.run("""
                        MATCH (d:Disease)-[:HAS_SYMPTOM]->(s:Symptom)
                        WHERE toLower(s.name) IN $symptoms
                        WITH d, collect(s.name) AS matched, count(s) AS cnt
                        ORDER BY cnt DESC
                        LIMIT 5
                        OPTIONAL MATCH (d)-[:TREATED_WITH]->(m:Medication)
                        OPTIONAL MATCH (d)-[:HAS_PRECAUTION]->(p:Precaution)
                        RETURN d.name AS disease, d.description AS description,
                               matched AS matching_symptoms, cnt AS match_count,
                               collect(DISTINCT m.name) AS medications,
                               collect(DISTINCT p.name) AS precautions
                    """, symptoms=[s.lower() for s in symptoms])
                    for r in symptom_match:
                        context_parts.append(f"\nDisease matching symptoms:\n{json.dumps(r.data(), indent=2, default=str)}")
    except Exception:
        pass

    return "\n\n".join(context_parts) if context_parts else "No context retrieved from graph."


def graph_rag_query(question):
    """Execute Graph RAG pipeline."""
    print(f"\n{'='*60}")
    print(f"Graph-Augmented RAG")
    print(f"{'='*60}")
    print(f"\n Pertanyaan: {question}\n")

    print(" Step 1: Retrieving context from graph...")
    graph_context = retrieve_graph_context(question)
    print(f"   -> Retrieved {len(graph_context)} chars of context")

    print("Step 2-3: Augmenting prompt & generating answer...\n")

    rag_system_prompt = """You are MediGraph AI, a medical knowledge assistant powered by a Neo4j graph database.
You answer medical questions using BOTH the graph context provided AND your medical knowledge.

IMPORTANT RULES:
1. Prioritize information from the Graph Context — it comes from a curated medical database.
2. If graph context is available, cite specific data points (disease names, medications, symptoms, etc.).
3. Structure your answer clearly with sections where appropriate.
4. If the graph context is empty or irrelevant, use your general medical knowledge but note this.
5. Always add a disclaimer that this is for educational purposes, not medical advice.
6. Answer in the same language as the question (Indonesian or English)."""

    rag_user_prompt = f"""GRAPH CONTEXT (from Neo4j Medical Knowledge Graph):
{graph_context}

USER QUESTION:
{question}

Please provide a comprehensive, well-structured answer based on the graph context above."""

    answer = tanya_system(rag_system_prompt, rag_user_prompt)

    print(f"{'─'*60}")
    print(f"MediGraph AI Answer:\n")
    print(answer)
    print(f"\n{'─'*60}")

    return {
        "question": question,
        "context": graph_context,
        "answer": answer
    }

---
## Cell 15 — Demo Interaktif
Masukkan pertanyaan medis apapun dan dapatkan jawaban cerdas dari **MediGraph AI** yang menggabungkan:
- **Text-to-Cypher** (query generation)
- **Graph Context** (retrieval)
- **LLM** (answer generation)

Jalankan cell di bawah, ketik pertanyaan, tekan Enter. Ketik `quit` untuk berhenti.

In [ ]:
# Interactive Q&A loop
print("MediGraph AI — Interactive Mode")
print("=" * 50)
print("Ketik pertanyaan medis dalam Bahasa Indonesia atau English.")
print("Ketik 'quit' atau 'exit' untuk berhenti.\n")

while True:
    try:
        user_input = input("Pertanyaan Anda: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nBye!")
        break

    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "q"]:
        print("\nTerima kasih telah menggunakan MediGraph AI!")
        break

    graph_rag_query(user_input)

MediGraph AI — Interactive Mode
Ketik pertanyaan medis dalam Bahasa Indonesia atau English.
Ketik 'quit' atau 'exit' untuk berhenti.

Pertanyaan Anda: my throat feels sore

Graph-Augmented RAG

 Pertanyaan: my throat feels sore

 Step 1: Retrieving context from graph...
   -> Retrieved 4584 chars of context
Step 2-3: Augmenting prompt & generating answer...

────────────────────────────────────────────────────────────
MediGraph AI Answer:

**Introduction**
A sore throat can be a symptom of various conditions, ranging from mild to severe. Based on the provided graph context, we will explore possible causes, their descriptions, and recommended treatments.

**Possible Causes**
According to the graph context, a sore throat can be associated with the following diseases:

1. **Seasonal allergies (hay fever)**: Characterized by allergic reactions to airborne allergens like pollen, causing sneezing, nasal congestion, itchy eyes, and throat irritation.
2. **Pneumonia**: An infection of the lung

---
##  Ringkasan Komponen

| No | Komponen | Cell | Tier |
|----|----------|------|------|
| 1 | Setup & Data Ingestion | Cell 1–7 | — |
| 2 | **LLM Text-to-Cypher** — Natural language -> Cypher query | Cell 8 | Tier 1 |
| 3 | **PageRank** — Identifikasi node paling penting | Cell 9 | Tier 1 |
| 4 | **Community Detection (Louvain)** — Clustering penyakit | Cell 10 | Tier 1 |
| 5 | **Shortest Path + Degree Centrality** — Path finding & centrality | Cell 11 | Tier 1 |
| 6 | **FastRP Embeddings + KNN** — ML node similarity | Cell 12 | Tier 2 |
| 7 | **LLM Graph Builder** — Extract entities from text -> populate graph | Cell 13 | Tier 3 |
| 8 | **Graph-Augmented RAG** — Retrieve + Augment + Generate | Cell 14 | Tier 4 |
| 9 | **Interactive Demo** — Full pipeline Q&A | Cell 15 | Demo |

### Tech Stack
- **Database**: Neo4j 5.x + GDS Plugin
- **LLM**: Moonshot Kimi K2 via OpenRouter
- **Language**: Python (Jupyter Notebook)
- **Libraries**: neo4j, openai, pandas

---
*MediGraph AI — Medical Knowledge Graph with Neo4j + LLM*